In [1]:
import os
import time
import json
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from IPython.display import clear_output
import matplotlib.colors as mcolors

In [2]:
# -----------------------------
# Load experiment metadata
# -----------------------------
exp_id = 8  # Change this to test different LSTM models

# Read results metadata to extract LSTM configuration
with open("results_metadata.json", "r") as f:
    metadata = pd.read_json(f)
row = metadata[metadata["experiment_id"] == exp_id].iloc[0]
PRED_LEN = int(row["pred_len"])
SEQ_LEN = 20
print(PRED_LEN)

25


In [3]:
# -----------------------------
# Load and prepare Zara dataset
# -----------------------------
df = pd.read_csv("converted_zara_1.csv")

PIXEL_TO_METER =0.05       # Conversion scale factor
df['x'] *= PIXEL_TO_METER
df['y'] *= PIXEL_TO_METER

# Build dataset with additional metadata (person_id, start_frame)
trajectories = []
for pid, person_df in df.groupby("person_id"):
    person_df = person_df.sort_values("frame_id")
    coords = person_df[['x', 'y']].values
    frames = person_df['frame_id'].values
    for i in range(len(coords) - SEQ_LEN - PRED_LEN):
        obs = coords[i:i+SEQ_LEN]
        fut = coords[i+SEQ_LEN:i+SEQ_LEN+PRED_LEN]
        start_frame = frames[i+SEQ_LEN]
        trajectories.append({
            "obs": obs,
            "fut": fut,
            "person_id": pid,
            "start_frame": int(start_frame)
        })

In [4]:
# -----------------------------
# Dataset wrapper for DataLoader
# -----------------------------
class TrajectoryDataset(Dataset):
    """
    Custom PyTorch dataset that returns:
    - observed sequence
    - future sequence (ground truth)
    - person_id
    - start frame index
    """
    def __init__(self, trajectories):
        self.trajectories = trajectories

    def __len__(self):
        return len(self.trajectories)

    def __getitem__(self, idx):
        item = self.trajectories[idx]
        obs = torch.tensor(item['obs'], dtype=torch.float32)
        fut = torch.tensor(item['fut'], dtype=torch.float32)
        return obs, fut, item['person_id'], item['start_frame']

# Create dataloader
data_loader = DataLoader(TrajectoryDataset(trajectories), batch_size=64, shuffle=False)


In [5]:
# -----------------------------
# LSTM Model Definition
# -----------------------------
class LSTMModel(nn.Module):
    """
    Basic LSTM network for trajectory prediction.
    Predicts 'output_len' number of (x, y) points from a sequence of inputs.
    """
    def __init__(self, input_size, hidden_size, output_len, num_layers, dropout, bidirectional):
        super(LSTMModel, self).__init__()
        self.bidirectional = bidirectional
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True, bidirectional=bidirectional
        )
        direction_multiplier = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_size * direction_multiplier, 2 * output_len)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        hn = torch.cat((hn[-2], hn[-1]), dim=1) if self.bidirectional else hn[-1]
        out = self.fc(hn)
        return out.view(-1, self.fc.out_features // 2, 2)

In [ ]:
# -----------------------------
# Load the trained model
# -----------------------------
model_path = f"lstm_zara_exp_{exp_id}.pth"
criterion = nn.MSELoss()
results = []

if os.path.exists(model_path):
    model = LSTMModel(
        input_size=2,
        hidden_size=int(row["hidden_size"]),
        output_len=int(row["pred_len"]),
        num_layers=int(row["num_layers"]),
        dropout=float(row["dropout"]),
        bidirectional=bool(row["bidirectional"])
    )
    model.load_state_dict(torch.load(model_path))
    model.eval()

    # -----------------------------
    # Evaluate and animate predictions
    # -----------------------------
    total_loss = 0
    total_ade = 0
    total_fde = 0
    count = 0
 
    all_preds, all_futs, all_obs = [], [], []
    all_pids, all_frames = [], []

    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            obs, fut, pids, frames = batch
            pred = model(obs)
            loss = criterion(pred, fut)
            total_loss += loss.item()

            #calculate average displacement error and final displacement error 
            ade = ((pred - fut) ** 2).sum(dim=2).sqrt().mean(dim=1).sum().item()
            fde = ((pred[:, -1, :] - fut[:, -1, :]) ** 2).sum(dim=1).sqrt().sum().item()
            total_ade += ade
            total_fde += fde
            count += fut.size(0)
            
            all_preds.append(pred)
            all_futs.append(fut)
            all_obs.append(obs)
            all_pids.extend(pids.numpy())
            all_frames.extend(frames.numpy())

    preds = torch.cat(all_preds)
    futs = torch.cat(all_futs)
    obs_all = torch.cat(all_obs)

    # Group predictions by frame ID
    frame_to_indices = {}
    for idx, frame in enumerate(all_frames):
        frame_to_indices.setdefault(frame, []).append(idx)

    #color_map = cm.get_cmap('hsv', 512)
    color_list = list(mcolors.TABLEAU_COLORS.values()) + list(mcolors.CSS4_COLORS.values())
    for frame_id, indices in sorted(frame_to_indices.items()):
        for t in range(PRED_LEN):
            clear_output(wait=True)
            #plt.figure(figsize=(6, 6))
            plt.figure(figsize=(12, 6))

            # Trajectory plot
            plt.subplot(1, 2, 1)
            
            for cidx, idx in enumerate(indices):
                #color = color_map(cidx * 20 % 512)
                color = color_list[cidx % len(color_list)]
                actual = futs[idx].numpy()
                predicted = preds[idx].numpy()
                observed = obs_all[idx].numpy()
                pid = all_pids[idx]

                plt.plot(observed[:, 0], observed[:, 1], linestyle='--', color=color, linewidth=1, label=f"P{pid} Obs")
                plt.plot(actual[:t+1, 0], actual[:t+1, 1], linestyle='-', color=color, linewidth=1.5, label=f"P{pid} A")
                plt.plot(predicted[:t+1, 0], predicted[:t+1, 1], linestyle=':', color=color, linewidth=2, label=f"P{pid} P")

            plt.title(f"Frame {frame_id}: Step {t+1}/{PRED_LEN}")
            #plt.xlim(-350, 350)
            #plt.ylim(-350, 100)
            #plt.xlabel("x")
            #plt.ylabel("y")
            plt.xlim(-10, 15)
            plt.ylim(-10, 5)
            plt.xlabel("x (meters)")
            plt.ylabel("y (meters)")          
            plt.legend(loc='upper right', fontsize='small', ncol=2)
            plt.grid(True)

            # Print numbers in the console or second plot
            plt.subplot(1, 2, 2)
            plt.axis('off')
            text_lines = []
            for cidx, idx in enumerate(indices):
                actual = futs[idx].numpy()
                predicted = preds[idx].numpy()
                observed = obs_all[idx].numpy()
                pid = all_pids[idx]
    
                actual_str = f"A: ({actual[t,0]:.2f}, {actual[t,1]:.2f})" if t < len(actual) else "A: ---"
                predicted_str = f"P: ({predicted[t,0]:.2f}, {predicted[t,1]:.2f})" if t < len(predicted) else "P: ---"
                text_lines.append(f"P{pid} | Obs End: ({observed[-1,0]:.2f}, {observed[-1,1]:.2f}) | {actual_str} | {predicted_str}")
    
            for i, line in enumerate(text_lines):
                plt.text(0.01, 0.95 - i*0.07, line, fontsize=10, transform=plt.gca().transAxes)
    
            plt.tight_layout()
            #plt.pause(0.1)

             # ADE/FDE Metrics plot
            plt.subplot(2, 2, 1)
            avg_ade = total_ade / count
            avg_fde = total_fde / count
            #plt.bar(['ADE', 'FDE'], [avg_ade, avg_fde], color=['skyblue', 'salmon'])
            #plt.title("Displacement Errors (m)")
            #plt.ylim(0, max(avg_ade, avg_fde) * 1.5)
            #plt.grid(axis='y')
            
            
            #plt.tight_layout()
            #plt.show()
            #time.sleep(0.05)
            
            avg_loss = total_loss / len(data_loader)
            results.append({**row.to_dict(), "zara2_loss": avg_loss})
            results.append({**row.to_dict(), "zara2_loss": avg_loss, "ADE": total_ade / count, "FDE": total_fde / count})       
